In [1]:
import numpy as np
import json
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel
from dotenv import load_dotenv
import re
import os


In [ ]:
# !pip install langchain langchain-google-genai
# !pip install python-dotenv


In [65]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("GOOGLE_API_KEY missing")

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0,
    api_key=api_key
)

emb = GoogleGenerativeAIEmbeddings(
    model="text-embedding-004",
    api_key=api_key
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [63]:
# print(llm.invoke("Say OK"))

In [66]:
emb.embed_query("Python developer with machine learning experience")

GoogleGenerativeAIError: Error embedding content (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [49]:
# Similarity Score
def cosine_sim(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def compute_similarity(resume_text: str, jd_text: str) -> int:
    e_resume = np.array(emb.embed_query(resume_text))
    e_jd = np.array(emb.embed_query(jd_text))
    sim = cosine_sim(e_resume, e_jd)
    score = round((sim + 1) / 2 * 100)
    return max(0, min(100, score))

In [50]:
# Extract top 10 JD skills (via LLM)
class SkillsOutput(BaseModel):
    skills: list[str]

skills_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """
You are an expert technical recruiter.

Extract at most 10 distinct, atomic, concrete skills
from the given job description.

Rules:
- Skills must be technologies, tools, frameworks, languages, or methodologies
- No soft skills or responsibilities
- Lowercase all skills
- Do NOT explain anything

Return ONLY valid JSON:
{
  "skills": [...]
}
"""),
    ("user", "{jd_text}")
])

def extract_jd_skills(jd_text: str) -> list[str]:
    chain = skills_prompt | llm.with_structured_output(SkillsOutput)
    result: SkillsOutput = chain.invoke({"jd_text": jd_text})
    return result.skills

In [51]:
SKILL_ALIASES = {
    "ml": "machine learning",
    "scikit learn": "scikit-learn",
    "tf": "tensorflow",
    "aws": "amazon web services",
}

In [52]:
# Match skills against resume
def _normalize(text: str) -> str:
    text = text.lower()
    for k, v in SKILL_ALIASES.items():
        text = text.replace(k, v)
    return re.sub(r"[^a-z0-9+#]", " ", text)

def _skill_present(skill: str, resume_norm: str) -> bool:
    pattern = r"\b" + re.escape(skill.lower()) + r"\b"
    return re.search(pattern, resume_norm) is not None

def match_skills(resume_text: str, skills: list[str]):
    resume_norm = _normalize(resume_text)
    in_resume = []
    missing = []
    for s in skills:
        if _skill_present(s, resume_norm):
            in_resume.append(s)
        else:
            missing.append(s)
    return in_resume, missing


In [53]:
# Wrapping these functions as LangChain TOOLS
from typing import Tuple

@tool
def compute_similarity_tool(resume_text: str, jd_text: str) -> int:
    """Compute a 0–100 similarity score between a resume and job description."""
    return compute_similarity(resume_text, jd_text)

@tool
def extract_jd_skills_tool(jd_text: str) -> list[str]:
    """Extract at most 10 key skills from a job description."""
    return extract_jd_skills(jd_text)

class MatchSkillsResult(BaseModel):
    skills_in_resume: list[str]
    skills_missing: list[str]

@tool
def match_skills_tool(resume_text: str, skills: list[str]) -> MatchSkillsResult:
    """
    Given resume text and a list of skills (from the JD),
    return which are present in the resume and which are missing.
    """
    in_resume, missing = match_skills(resume_text, skills)
    return MatchSkillsResult(skills_in_resume=in_resume, skills_missing=missing)


In [54]:
# Simple fixed pipeline
def analyze_resume(resume_text: str, jd_text: str):
    score = compute_similarity(resume_text, jd_text)
    jd_skills = extract_jd_skills(jd_text)
    in_resume, missing = match_skills(resume_text, jd_skills)
    return {
    "semantic_similarity_score": score,
    "jd_skills": jd_skills,
    "skills_in_resume": in_resume,
    "skills_missing": missing,
}

In [12]:
tools = [compute_similarity_tool, extract_jd_skills_tool, match_skills_tool]

agent_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a recruitment assistant agent. "
     "Given a resume and job description, you MUST: "
     "1) call compute_similarity_tool, "
     "2) call extract_jd_skills_tool, "
     "3) call match_skills_tool, "
     "then return a JSON summary."),
    ("user", "{input}"),
    ("assistant", "Think step by step and use the tools when needed.")
])

chain = agent_prompt | llm.bind_tools(tools)

def run_agent(resume_text: str, jd_text: str):
    input_text = (
        "Here is the resume:\n"
        f"{resume_text}\n\n"
        "Here is the job description:\n"
        f"{jd_text}"
    )

    response = chain.invoke({"input": input_text})
    return response


In [13]:
resume_txt = """
Name: Ahmed Hassan
Location: Dubai, UAE
Email: ahmed.hassan@email.com

Phone: +971-XX-XXX-XXXX

Professional Summary

Results-oriented Digital Marketing Professional with over 3 years of experience managing paid advertising, SEO strategies, and social media campaigns. Proven ability to increase website traffic, improve conversion rates, and deliver measurable ROI for marketing initiatives.

Work Experience
Digital Marketing Executive

ABC Trading LLC — Dubai, UAE
Jan 2023 – Present

Managed Google Ads and Meta Ads campaigns with monthly budgets exceeding AED 50,000

Improved website organic traffic by 35% through SEO optimization

Created and managed email marketing campaigns with average open rates of 28%

Prepared weekly and monthly performance reports using Google Analytics

Collaborated with designers to develop high-converting landing pages

Marketing Assistant

XYZ Solutions — Dubai, UAE
Jun 2021 – Dec 2022

Supported digital marketing campaigns across social media and search platforms

Assisted with keyword research and on-page SEO optimization

Scheduled and managed social media posts

Tracked campaign metrics and prepared basic performance dashboards

Education

Bachelor of Business Administration (Marketing)
University of Sharjah — UAE
2017 – 2021

Skills

Google Ads & Meta Ads

Google Analytics

SEO & SEM

Email Marketing (Mailchimp, HubSpot)

Social Media Management

Microsoft Excel & Google Sheets

A/B Testing & Conversion Rate Optimization

Certifications

Google Ads Search Certification

HubSpot Content Marketing Certification
"""

jd_txt = """
Job Title: Digital Marketing Specialist
Job Summary

We are seeking a results-driven Digital Marketing Specialist to plan, execute, and optimize online marketing campaigns across multiple digital channels. The ideal candidate will have strong analytical skills, creativity, and hands-on experience with paid advertising, SEO, and content marketing.

Key Responsibilities

Plan and manage digital marketing campaigns across Google Ads, Meta Ads, and email platforms

Optimize website content for SEO and improve organic search rankings

Analyze campaign performance and prepare monthly performance reports

Manage social media accounts and content calendars

Conduct A/B testing to improve conversion rates

Collaborate with design and content teams to develop marketing materials

Track KPIs such as CTR, CPA, and ROI

Required Skills & Qualifications

Bachelor’s degree in Marketing, Business, or related field

2+ years of experience in digital marketing

Experience with Google Analytics, Google Ads, and Meta Business Manager

Strong understanding of SEO and SEM

Excellent communication and project management skills

Proficiency in Microsoft Excel or Google Sheets
"""

In [14]:
res = run_agent(resume_txt, jd_txt)

In [46]:
res2 = analyze_resume(resume_txt, jd_txt)

GoogleGenerativeAIError: Error embedding content (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/embedding-001 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [15]:
res

AIMessage(content='\n\n1.  **Call `compute_similarity_tool`**: Calculate the overall similarity score between the resume and the job description.\n2.  **Call `extract_jd_skills_tool`**: Extract the key skills from the job description.\n3.  **Call `match_skills_tool`**: Check which of the extracted skills are present in the resume.\n4.  **Format the final JSON summary**: Combine the results from all three steps into the required JSON structure.\n\n**Step 1: Compute Similarity**\nCalling `default_api:compute_similarity_tool` with the provided resume and JD texts.\n\n**Step 2: Extract JD Skills**\nCalling `default_api:extract_jd_skills_tool` with the JD text.\n\n**Step 3: Match Skills**\nThe skills extracted in Step 2 will be used as input for `default_api:match_skills_tool` along with the resume text.\n\n**Step 4: Final Output**\nThe final output will be a JSON object containing the similarity score, the extracted skills, and the skill match results.', additional_kwargs={'function_call':

In [17]:
res2

{'similarity_score': 87,
 'top_skills': ['Google Ads',
  'Meta Ads',
  'SEO',
  'SEM',
  'Google Analytics',
  'Email Marketing',
  'Social Media Management',
  'A/B Testing',
  'Data Analysis',
  'Microsoft Excel'],
 'skills_in_resume': ['Google Ads',
  'Meta Ads',
  'SEO',
  'SEM',
  'Google Analytics',
  'Email Marketing',
  'Social Media Management',
  'Microsoft Excel'],
 'skills_missing': ['A/B Testing', 'Data Analysis']}

In [3]:
# !pip install ipywidgets pdfplumber python-docx

In [55]:
import ipywidgets as widgets
from IPython.display import display, clear_output

import pdfplumber
import docx

In [56]:
def extract_text_from_file(uploaded_file):
    """
    uploaded_file: tuple (filename, file_bytes)
    returns: extracted text as string
    """
    filename, file_bytes = uploaded_file
    filename = filename.lower()

    if filename.endswith(".pdf"):
        text = ""
        with pdfplumber.open(file_bytes) as pdf:
            for page in pdf.pages:
                text += page.extract_text() + "\n"
        return text

    elif filename.endswith(".docx"):
        doc = docx.Document(file_bytes)
        return "\n".join([para.text for para in doc.paragraphs])

    elif filename.endswith(".txt"):
        return file_bytes.read().decode("utf-8")

    else:
        raise ValueError("Unsupported file format. Upload PDF, DOCX, or TXT.")


In [57]:
jd_input = widgets.Textarea(
    value=jd_txt,  # you can pre-fill it
    placeholder="Paste job description here...",
    description="Job Description:",
    layout=widgets.Layout(width="100%", height="200px")
)


In [58]:
resume_uploader = widgets.FileUpload(
    accept=".pdf,.docx,.txt",
    multiple=False,
    description="Upload Resume"
)


In [59]:
analyze_button = widgets.Button(
    description="Analyze Resume",
    button_style="primary"
)

output = widgets.Output()


In [60]:
def on_analyze_clicked(b):
    with output:
        clear_output()

        if not resume_uploader.value:
            print("❌ Please upload a resume.")
            return

        # Get uploaded file safely
        filename, uploaded_file = next(iter(resume_uploader.value.items()))
        file_bytes = uploaded_file["content"]

        print(f"📄 Uploaded file: {filename}")

        # Extract text
        try:
            resume_text = extract_text_from_file((filename, BytesIO(file_bytes)))
        except Exception as e:
            print(f"Error reading file: {e}")
            return

        jd_text = jd_input.value
        if not jd_text.strip():
            print("❌ Please enter a job description.")
            return

        result = analyze_resume(resume_text, jd_text)

        print("✅ Analysis complete\n")
        print(result)


In [42]:
display(
    widgets.VBox([
        jd_input,
        resume_uploader,
        analyze_button,
        output
    ])
)


In [61]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from io import BytesIO

resume_uploader = widgets.FileUpload(
    accept=".pdf,.docx,.txt",
    multiple=False,
    description="Upload Resume"
)

jd_input = widgets.Textarea(
    description="Job Description",
    layout=widgets.Layout(width="100%", height="150px")
)

analyze_button = widgets.Button(
    description="Analyze Resume",
    button_style="primary"
)

output = widgets.Output()

def on_analyze_clicked(b):
    with output:
        clear_output()

        if not resume_uploader.value:
            print("❌ Please upload a resume.")
            return

        filename, uploaded_file = next(iter(resume_uploader.value.items()))
        file_bytes = uploaded_file["content"]

        print(f"📄 Uploaded file: {filename}")

        try:
            resume_text = extract_text_from_file((filename, BytesIO(file_bytes)))
        except Exception as e:
            print(f"Error reading file: {e}")
            return

        jd_text = jd_input.value
        if not jd_text.strip():
            print("❌ Please enter a job description.")
            return

        result = analyze_resume(resume_text, jd_text)

        print("✅ Analysis complete\n")
        print(result)

analyze_button.on_click(on_analyze_clicked)

# 👇 THIS IS THE KEY PART
display(resume_uploader, jd_input, analyze_button, output)


FileUpload(value={}, accept='.pdf,.docx,.txt', description='Upload Resume')

Textarea(value='', description='Job Description', layout=Layout(height='150px', width='100%'))

Button(button_style='primary', description='Analyze Resume', style=ButtonStyle())

Output()